# Step 3 Full Project Workbench

This notebook consolidates the full `STEP-3` project into one place.
It does **not** duplicate the production logic into brittle notebook copies. Instead, it imports or wraps the existing Step 3 scripts so you can drive the whole project from one notebook surface.

## Covered Workflows
- Legacy red-flag engine: `train.py`, `predict.py`, `transaction_classifier.py`, `feature_engine.py`, `red_flag_detector.py`
- Home Credit baseline: `home_credit_task3.py`
- Improved Home Credit model and master-dataset test: `home_credit_improved.py`, `test_on_master.py`
- Synthetic gig-worker model: `gig_credit_model.py`
- Single-score master-dataset scorer: `gigscore_model.py`
- Dual-model pipeline: `dual_model_pipeline.py`
- Behavioral three-zone model: `test_on_master_gig.py`
- Report chart generation: `generate_report_charts.py`

Heavy sections are split into independent cells so you can run only the parts you need.


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import json
import subprocess
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)


def locate_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "STEP-3").exists():
            return candidate
    raise FileNotFoundError("Could not locate a project root that contains STEP-3/")


def first_existing(*paths: Path | str | None) -> Path | None:
    for raw_path in paths:
        if raw_path is None:
            continue
        path = Path(raw_path).expanduser()
        if path.exists():
            return path
    return None


def load_step3_module(filename: str, module_name: str | None = None):
    path = STEP3_DIR / filename
    name = module_name or f"step3_{path.stem}".replace("-", "_")
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None, f"Unable to load {path}"
    spec.loader.exec_module(module)
    return module


def run_cli(script_name: str, *args: str):
    cmd = [sys.executable, str(STEP3_DIR / script_name), *map(str, args)]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
STEP3_DIR = PROJECT_ROOT / "STEP-3"
STEP3_DATA_DIR = STEP3_DIR / "data"
STEP3_MODELS_DIR = STEP3_DIR / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RESULTS_DIR = PROJECT_ROOT / "results"
DOWNLOADS_DIR = Path.home() / "Downloads"

for path in (PROJECT_ROOT, STEP3_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

MASTER_DATASET_PATH = first_existing(
    DOWNLOADS_DIR / "GigScore_Master_Dataset (1).csv",
    DOWNLOADS_DIR / "GigScore_Master_Dataset.csv",
    PROJECT_ROOT / "GigScore_Master_Dataset.csv",
    PROJECT_ROOT / "data" / "GigScore_Master_Dataset.csv",
)

HOME_CREDIT_FILES = {
    "application_train": first_existing(
        DOWNLOADS_DIR / "application_train.csv",
        DOWNLOADS_DIR / "gigtanush" / "application_train.csv",
    ),
    "pos_cash": first_existing(DOWNLOADS_DIR / "POS_CASH_balance.csv"),
    "credit_card": first_existing(DOWNLOADS_DIR / "credit_card_balance.csv"),
    "installments": first_existing(DOWNLOADS_DIR / "installments_payments.csv"),
}

LEGACY_TC_MODEL = STEP3_MODELS_DIR / "transaction_classifier.joblib"
LEGACY_RFD_MODEL = STEP3_MODELS_DIR / "red_flag_detector.joblib"

HC_V1_MODEL = STEP3_MODELS_DIR / "home_credit_risk_model.joblib"
HC_V1_RESULTS = RESULTS_DIR / "task3_home_credit_validation_results.csv"
HC_V1_METRICS = RESULTS_DIR / "task3_home_credit_metrics.json"

HC_V2_MODEL = STEP3_MODELS_DIR / "home_credit_risk_model_v2.joblib"
HC_V2_RESULTS = RESULTS_DIR / "task3_improved_validation_results.csv"
HC_V2_METRICS = RESULTS_DIR / "task3_improved_metrics.json"

GIG_WORKER_MODEL = STEP3_MODELS_DIR / "gig_credit_model.joblib"
GIG_WORKER_RESULTS = RESULTS_DIR / "gig_model_validation_results.csv"
GIG_WORKER_METRICS = RESULTS_DIR / "gig_model_metrics.json"

THREE_ZONE_MODEL = STEP3_MODELS_DIR / "gig_credit_model_master.joblib"
THREE_ZONE_RESULTS = RESULTS_DIR / "gig_master_zone_predictions.csv"
THREE_ZONE_METRICS = RESULTS_DIR / "gig_master_zone_metrics.json"
THREE_ZONE_REPORT = RESULTS_DIR / "gig_master_zone_report.txt"
THREE_ZONE_PAYLOAD = RESULTS_DIR / "gig_master_agent4_payload.json"

MASTER_PREDICTIONS = RESULTS_DIR / "master_dataset_predictions.csv"
CHARTS_DIR = RESULTS_DIR / "charts"

path_status = pd.DataFrame(
    [
        {"asset": "project_root", "path": str(PROJECT_ROOT), "exists": PROJECT_ROOT.exists()},
        {"asset": "step3_data_dir", "path": str(STEP3_DATA_DIR), "exists": STEP3_DATA_DIR.exists()},
        {"asset": "master_dataset", "path": str(MASTER_DATASET_PATH) if MASTER_DATASET_PATH else "<missing>", "exists": MASTER_DATASET_PATH is not None},
        {"asset": "application_train", "path": str(HOME_CREDIT_FILES['application_train']) if HOME_CREDIT_FILES['application_train'] else "<missing>", "exists": HOME_CREDIT_FILES['application_train'] is not None},
        {"asset": "POS_CASH_balance", "path": str(HOME_CREDIT_FILES['pos_cash']) if HOME_CREDIT_FILES['pos_cash'] else "<missing>", "exists": HOME_CREDIT_FILES['pos_cash'] is not None},
        {"asset": "credit_card_balance", "path": str(HOME_CREDIT_FILES['credit_card']) if HOME_CREDIT_FILES['credit_card'] else "<missing>", "exists": HOME_CREDIT_FILES['credit_card'] is not None},
        {"asset": "installments_payments", "path": str(HOME_CREDIT_FILES['installments']) if HOME_CREDIT_FILES['installments'] else "<missing>", "exists": HOME_CREDIT_FILES['installments'] is not None},
    ]
)

display(path_status)
PROJECT_ROOT


,asset,path,exists
0,project_root,/Users/adittsingh/GigScore,True
1,step3_data_dir,/Users/adittsingh/GigScore/STEP-3/data,True
2,master_dataset,/Users/adittsingh/Downloads/GigScore_Master_Da...,True
3,application_train,/Users/adittsingh/Downloads/application_train.csv,True
4,POS_CASH_balance,/Users/adittsingh/Downloads/POS_CASH_balance.csv,True
5,credit_card_balance,/Users/adittsingh/Downloads/credit_card_balanc...,True
6,installments_payments,/Users/adittsingh/Downloads/installments_payme...,True


PosixPath('/Users/adittsingh/GigScore')

In [2]:
inventory = pd.DataFrame(
    [
        {
            "workflow": "Legacy red-flag engine",
            "entry_points": "train.py, predict.py",
            "primary_inputs": "STEP-3/data/*.csv",
            "primary_outputs": "transaction_classifier.joblib, red_flag_detector.joblib",
        },
        {
            "workflow": "Home Credit baseline",
            "entry_points": "home_credit_task3.py",
            "primary_inputs": "application_train + POS/CCB/installments",
            "primary_outputs": "home_credit_risk_model.joblib, task3_home_credit_*.json/csv",
        },
        {
            "workflow": "Improved Home Credit",
            "entry_points": "home_credit_improved.py, test_on_master.py",
            "primary_inputs": "application_train + POS/CCB/installments + master dataset",
            "primary_outputs": "home_credit_risk_model_v2.joblib, task3_improved_*.json/csv",
        },
        {
            "workflow": "Synthetic gig-worker model",
            "entry_points": "gig_credit_model.py",
            "primary_inputs": "synthetic transactions generated in-process",
            "primary_outputs": "gig_credit_model.joblib, gig_model_metrics.json",
        },
        {
            "workflow": "Single-score master model",
            "entry_points": "gigscore_model.py",
            "primary_inputs": "master dataset",
            "primary_outputs": "gigscore_model.joblib, master_dataset_predictions.csv",
        },
        {
            "workflow": "Dual-model pipeline",
            "entry_points": "dual_model_pipeline.py",
            "primary_inputs": "master dataset",
            "primary_outputs": "dual_model_pipeline.joblib, dual_model_metrics.json",
        },
        {
            "workflow": "Behavioral three-zone model",
            "entry_points": "test_on_master_gig.py",
            "primary_inputs": "master dataset",
            "primary_outputs": "gig_credit_model_master.joblib, gig_master_zone_*.json/csv/txt",
        },
        {
            "workflow": "Report charts",
            "entry_points": "generate_report_charts.py",
            "primary_inputs": "results/master_dataset_predictions.csv",
            "primary_outputs": "results/charts/*.png",
        },
    ]
)

display(inventory)


,workflow,entry_points,primary_inputs,primary_outputs
0,Legacy red-flag engine,"train.py, predict.py",STEP-3/data/*.csv,"transaction_classifier.joblib, red_flag_detect..."
1,Home Credit baseline,home_credit_task3.py,application_train + POS/CCB/installments,"home_credit_risk_model.joblib, task3_home_cred..."
2,Improved Home Credit,"home_credit_improved.py, test_on_master.py",application_train + POS/CCB/installments + mas...,"home_credit_risk_model_v2.joblib, task3_improv..."
3,Synthetic gig-worker model,gig_credit_model.py,synthetic transactions generated in-process,"gig_credit_model.joblib, gig_model_metrics.json"
4,Single-score master model,gigscore_model.py,master dataset,"gigscore_model.joblib, master_dataset_predicti..."
5,Dual-model pipeline,dual_model_pipeline.py,master dataset,"dual_model_pipeline.joblib, dual_model_metrics..."
6,Behavioral three-zone model,test_on_master_gig.py,master dataset,"gig_credit_model_master.joblib, gig_master_zon..."
7,Report charts,generate_report_charts.py,results/master_dataset_predictions.csv,results/charts/*.png


## Usage Notes

- Run the setup cell first.
- Each section below is independent.
- External-data sections skip cleanly when the needed CSVs are not present.
- This notebook keeps the underlying scripts as the source of truth, so fixing a model in `STEP-3/*.py` automatically updates the notebook behavior too.


## 1. Legacy Red-Flag Engine

This section covers the original transaction-classification and rule-plus-ML red-flag pipeline that powers `train.py` and `predict.py`.


In [3]:
import config
transaction_classifier_mod = importlib.import_module("transaction_classifier")
feature_engine_mod = importlib.import_module("feature_engine")
red_flag_detector_mod = importlib.import_module("red_flag_detector")
predict_mod = load_step3_module("predict.py", "step3_predict_nb")

config.DATA_DIR = str(STEP3_DATA_DIR)
config.MODEL_DIR = str(STEP3_MODELS_DIR)

profiles = pd.read_csv(STEP3_DATA_DIR / config.USER_PROFILES_CSV)
txns = pd.read_csv(STEP3_DATA_DIR / config.BANK_STATEMENTS_CSV)

legacy_overview = pd.DataFrame(
    {
        "metric": ["users", "transactions", "labeled_categories", "default_rate"],
        "value": [
            len(profiles),
            len(txns),
            int(txns["category_actual"].notna().sum()),
            round(float(profiles["default_label"].mean()), 4),
        ],
    }
)

display(legacy_overview)
txns.head()


,metric,value
0,users,800.0000
1,transactions,304164.0000
2,labeled_categories,304164.0000
3,default_rate,0.3187


,transaction_id,user_id,date,narration,amount,type,running_balance,category_actual,merchant_category,counterparty_id
0,TXN00000001,USR0001,2024-01-01 19:20:55,UPI/413333214298/Pay/ZEPTO/ZEPTO_MERCHANT/PNB,539.0,CREDIT,7746.0,GIG_INCOME,Other,ZEPTO_PAYOUT
1,TXN00000002,USR0001,2024-01-01 19:54:00,UPI/565153861212/Payment/MEESHO/MEESHO_ORDER/PNB,1820.0,DEBIT,5926.0,ECOMMERCE,Shopping,MEESHO
2,TXN00000003,USR0001,2024-01-01 19:55:56,UPI/121441623622/Payment/BLINKIT/BLINKIT_MERCH...,556.0,CREDIT,6482.0,GIG_INCOME,Other,BLINKIT_PAYOUT
3,TXN00000004,USR0001,2024-01-01 20:40:19,UPI/402545937444//DREAM11/DREAM11_APP/PNB,50.0,DEBIT,6432.0,GAMBLING,Entertainment,DREAM11
4,TXN00000005,USR0001,2024-01-02 20:14:49,UPI/287815708381//BLINKIT/BLINKIT_MERCHANT/PNB,501.0,CREDIT,6933.0,GIG_INCOME,Other,BLINKIT_PAYOUT


In [4]:
TransactionClassifier = transaction_classifier_mod.TransactionClassifier
RedFlagDetector = red_flag_detector_mod.RedFlagDetector
extract_all_users = feature_engine_mod.extract_all_users

tc = TransactionClassifier()
tc_report = tc.train(txns)
tc.save(str(LEGACY_TC_MODEL))

legacy_features = extract_all_users(txns, profiles)
legacy_labels = profiles.set_index("user_id").loc[legacy_features.index, "default_label"]

rfd = RedFlagDetector()
rfd_report = rfd.train(legacy_features, legacy_labels)
rfd.save(str(LEGACY_RFD_MODEL))

legacy_summary = pd.DataFrame(
    [
        {
            "transaction_classifier_accuracy": round(float(tc_report.get("accuracy", 0.0)), 4),
            "red_flag_auc_roc": round(float(rfd_report.get("auc_roc", 0.0)), 4),
            "saved_tc_model": str(LEGACY_TC_MODEL),
            "saved_rfd_model": str(LEGACY_RFD_MODEL),
        }
    ]
)
top_legacy_features = pd.DataFrame(
    list(rfd_report.get("feature_importances", {}).items()),
    columns=["feature", "importance"],
)

display(legacy_summary)
top_legacy_features.head(15)


Transaction classifier saved to /Users/adittsingh/GigScore/STEP-3/models/transaction_classifier.joblib
Red flag detector saved to /Users/adittsingh/GigScore/STEP-3/models/red_flag_detector.joblib


,transaction_classifier_accuracy,red_flag_auc_roc,saved_tc_model,saved_rfd_model
0,1.0,0.9777,/Users/adittsingh/GigScore/STEP-3/models/trans...,/Users/adittsingh/GigScore/STEP-3/models/red_f...


,feature,importance
0,gambling_spend_ratio,0.3673
1,rent_late_ratio,0.0605
2,rent_day_std,0.0319
3,emergency_credit_count,0.0285
4,cash_max_to_income,0.0262
5,gambling_late_night_ratio,0.0232
6,balance_cv,0.0232
7,p2p_in_total,0.0226
8,top_category_share,0.0218
9,late_night_total_spent,0.0201


In [5]:
demo_user_id = profiles.iloc[0]["user_id"]
demo_profile = profiles.set_index("user_id").loc[demo_user_id]
demo_txns = txns[txns["user_id"] == demo_user_id].copy()

legacy_demo = predict_mod.assess_applicant(
    demo_txns,
    tc,
    rfd,
    monthly_income=float(demo_profile["monthly_income_approx"]),
)

legacy_demo_summary = pd.Series(
    {
        "user_id": demo_user_id,
        "risk_score": legacy_demo["risk_score"],
        "risk_level": legacy_demo["risk_level"],
        "flagged": legacy_demo["flagged"],
        "red_flag_count": len(legacy_demo["red_flags"]),
        "observation_count": len(legacy_demo["observations"]),
    }
).to_frame("value")

display(legacy_demo_summary)
pd.DataFrame(legacy_demo["red_flags"] or legacy_demo["observations"]).head(10)


,value
user_id,USR0001
risk_score,0.099
risk_level,LOW
flagged,False
red_flag_count,0
observation_count,11


,name,category,probability,reason,severity
0,gambling_frequency,GAMBLING,0.707,Gambling transactions 7.2/month (threshold: 3),MEDIUM
1,gambling_spend_ratio,GAMBLING,0.723,Gambling spend is 19.2% of income (threshold: 5%),MEDIUM
2,gambling_escalation,GAMBLING,0.560,Gambling amounts escalating by 88% over time,MEDIUM
3,gambling_multi_platform,GAMBLING,0.723,Active on 12 gambling platforms,MEDIUM
4,rent_late_payment,RENT,0.255,Rent paid late 25% of months (after day 10),MEDIUM
5,rent_irregular_amount,RENT,0.370,Rent amount variance 39% (threshold: 30%),MEDIUM
6,rent_overburden,RENT,0.625,Rent consumes 95% of income,MEDIUM
7,large_cash_withdrawal,CASH,0.723,Single cash withdrawal = 95% of monthly income,MEDIUM
8,frequent_negative_balance,BALANCE,0.723,Account negative 80% of transactions,MEDIUM
9,overspending,BALANCE,0.723,Monthly spend is 567% of income,MEDIUM


## 2. Home Credit Baseline

This runs the balance-table-only baseline from `home_credit_task3.py` when the Home Credit CSVs are available.


In [6]:
home_credit_v1 = load_step3_module("home_credit_task3.py", "step3_home_credit_v1_nb")
hc_v1_ready = all(HOME_CREDIT_FILES.values())

if not hc_v1_ready:
    print("Skipping Home Credit baseline: one or more source files are missing.")
else:
    hc_v1_matrix = home_credit_v1.build_feature_matrix(
        application_path=str(HOME_CREDIT_FILES["application_train"]),
        pos_path=str(HOME_CREDIT_FILES["pos_cash"]),
        credit_card_path=str(HOME_CREDIT_FILES["credit_card"]),
        installments_path=str(HOME_CREDIT_FILES["installments"]),
        chunksize=500_000,
    )
    hc_v1_metrics = home_credit_v1.train_and_score(
        data=hc_v1_matrix,
        model_path=str(HC_V1_MODEL),
        results_path=str(HC_V1_RESULTS),
        metrics_path=str(HC_V1_METRICS),
        validation_size=0.2,
        threshold=0.5,
    )

    display(
        pd.DataFrame(
            [
                {
                    "rows": hc_v1_matrix.shape[0],
                    "features": hc_v1_matrix.shape[1] - 1,
                    "roc_auc": round(hc_v1_metrics["roc_auc"], 4),
                    "avg_precision": round(hc_v1_metrics["average_precision"], 4),
                }
            ]
        )
    )
    display(pd.read_csv(HC_V1_RESULTS).head(10))



[APPLICATION] Loaded 307,511 labeled applicants

[POS_CASH] Aggregating /Users/adittsingh/Downloads/POS_CASH_balance.csv
  processed 2,500,000 rows
  processed 5,000,000 rows
  processed 7,500,000 rows
  processed 10,000,000 rows
  completed 10,001,358 rows -> 337,252 applicants

[CREDIT_CARD] Aggregating /Users/adittsingh/Downloads/credit_card_balance.csv
  processed 2,500,000 rows
  completed 3,840,312 rows -> 103,558 applicants

[INSTALLMENTS] Aggregating /Users/adittsingh/Downloads/installments_payments.csv
  processed 2,500,000 rows
  processed 5,000,000 rows
  processed 7,500,000 rows
  processed 10,000,000 rows
  processed 12,500,000 rows
  completed 13,605,401 rows -> 339,587 applicants

[TRAIN] Fitting XGBoost on 246,008 train rows


,rows,features,roc_auc,avg_precision
0,307511,111,0.6591,0.1629


,SK_ID_CURR,actual_target,predicted_probability,predicted_label,risk_band,pos_overdue_ratio,pos_dpd_max,ccb_utilization_mean,ccb_dpd_max,inst_late_ratio,inst_underpaid_ratio
0,204761,1,0.923455,1,HIGH,0.071429,32.0,0.984952,1.0,0.182353,0.208092
1,159610,0,0.912770,1,HIGH,0.142857,22.0,0.000000,0.0,0.538462,0.384615
2,135439,0,0.902867,1,HIGH,0.000000,0.0,0.950771,0.0,0.083333,0.055556
3,448141,0,0.897711,1,HIGH,0.000000,0.0,0.947586,0.0,0.230769,0.153846
4,180988,0,0.895205,1,HIGH,0.000000,0.0,0.927652,0.0,0.147059,0.000000
5,271820,0,0.894645,1,HIGH,0.047619,19.0,0.911141,0.0,0.255319,0.085106
6,283823,1,0.892631,1,HIGH,0.037313,11.0,0.000000,0.0,0.195804,0.272109
7,443619,0,0.892104,1,HIGH,0.125000,57.0,0.000000,0.0,0.000000,0.161290
8,395780,1,0.890850,1,HIGH,0.000000,0.0,0.614598,0.0,0.138889,0.111111
9,270544,0,0.889478,1,HIGH,0.230769,14.0,0.000000,0.0,0.378378,0.540541


## 3. Improved Home Credit Model

This section uses the richer application + balance-table feature set from `home_credit_improved.py`.


In [7]:
home_credit_v2 = load_step3_module("home_credit_improved.py", "step3_home_credit_v2_nb")
hc_v2_ready = all(HOME_CREDIT_FILES.values())

if not hc_v2_ready:
    print("Skipping improved Home Credit model: one or more source files are missing.")
else:
    hc_v2_app = home_credit_v2.load_application_features(str(HOME_CREDIT_FILES["application_train"]))
    hc_v2_pos = home_credit_v2.aggregate_pos_cash(str(HOME_CREDIT_FILES["pos_cash"]), 500_000)
    hc_v2_ccb = home_credit_v2.aggregate_credit_card(str(HOME_CREDIT_FILES["credit_card"]), 500_000)
    hc_v2_inst = home_credit_v2.aggregate_installments(str(HOME_CREDIT_FILES["installments"]), 500_000)

    hc_v2_data = hc_v2_app.join(hc_v2_pos, how="left").join(hc_v2_ccb, how="left").join(hc_v2_inst, how="left").fillna(0)
    hc_v2_metrics = home_credit_v2.train_model(
        data=hc_v2_data,
        model_path=str(HC_V2_MODEL),
        results_path=str(HC_V2_RESULTS),
        metrics_path=str(HC_V2_METRICS),
        threshold=0.5,
    )

    display(
        pd.DataFrame(
            [
                {
                    "rows": hc_v2_data.shape[0],
                    "features": hc_v2_data.shape[1] - 1,
                    "cv_auc_mean": round(hc_v2_metrics["cv_auc_mean"], 4),
                    "test_roc_auc": round(hc_v2_metrics["test_roc_auc"], 4),
                    "test_avg_precision": round(hc_v2_metrics["test_average_precision"], 4),
                }
            ]
        )
    )
    display(pd.read_csv(HC_V2_RESULTS).head(10))



[APPLICATION] Loading /Users/adittsingh/Downloads/application_train.csv
  Loaded 307,511 applicants, 121 raw columns
  Engineered 70 application features

[POS_CASH] Aggregating /Users/adittsingh/Downloads/POS_CASH_balance.csv
  Loaded 10,001,358 rows
  Generated 27 POS features for 337,252 applicants

[CREDIT_CARD] Aggregating /Users/adittsingh/Downloads/credit_card_balance.csv
  Loaded 3,840,312 rows
  Generated 34 credit card features for 103,558 applicants

[INSTALLMENTS] Aggregating /Users/adittsingh/Downloads/installments_payments.csv
  Loaded 13,605,401 rows
  Generated 30 installment features for 339,587 applicants

[CV] Running 5-fold stratified cross-validation...
  Fold 1: AUC = 0.7820
  Fold 2: AUC = 0.7756
  Fold 3: AUC = 0.7835
  Fold 4: AUC = 0.7776
  Fold 5: AUC = 0.7862
  OOF AUC: 0.7809 (mean fold: 0.7810 ± 0.0039)

[TRAIN] Fitting final model on 261,384 rows...

[TEST] Held-out test results (46,127 rows):
  ROC-AUC:           0.7843
  Average Precision: 0.2811


,rows,features,cv_auc_mean,test_roc_auc,test_avg_precision
0,307511,161,0.781,0.7843,0.2811


,SK_ID_CURR,actual_target,predicted_probability,predicted_label,risk_band
0,427557,0,0.956375,1,HIGH
1,214692,1,0.956109,1,HIGH
2,453154,1,0.955403,1,HIGH
3,395476,1,0.953408,1,HIGH
4,163956,1,0.949513,1,HIGH
5,426909,1,0.946167,1,HIGH
6,247181,0,0.945921,1,HIGH
7,154759,1,0.945317,1,HIGH
8,450010,1,0.943214,1,HIGH
9,373606,1,0.943180,1,HIGH


## 4. Test Improved Home Credit Model on the Master Dataset

This mirrors the logic in `test_on_master.py`.


In [8]:
test_on_master_mod = load_step3_module("test_on_master.py", "step3_test_on_master_nb")

if MASTER_DATASET_PATH is None:
    print("Skipping master-dataset test: master dataset not found.")
elif not HC_V2_MODEL.exists():
    print(f"Skipping master-dataset test: missing model bundle at {HC_V2_MODEL}")
else:
    hc_v2_bundle = joblib.load(HC_V2_MODEL)
    hc_v2_master_X, hc_v2_master_y = test_on_master_mod.load_and_prepare(
        str(MASTER_DATASET_PATH),
        hc_v2_bundle["feature_columns"],
    )
    hc_v2_master_proba = hc_v2_bundle["model"].predict_proba(hc_v2_master_X)[:, 1]
    hc_v2_master_threshold = float(hc_v2_bundle.get("threshold", 0.5))
    hc_v2_master_pred = (hc_v2_master_proba >= hc_v2_master_threshold).astype(int)
    hc_v2_master_metrics = test_on_master_mod.performance_report(
        hc_v2_master_y,
        hc_v2_master_proba,
        hc_v2_master_pred,
        hc_v2_master_threshold,
    )

    master_summary = pd.DataFrame(
        [
            {
                "rows": len(hc_v2_master_y),
                "roc_auc": round(hc_v2_master_metrics["roc_auc"], 4),
                "avg_precision": round(hc_v2_master_metrics["avg_precision"], 4),
                "precision": round(hc_v2_master_metrics["precision"], 4),
                "recall": round(hc_v2_master_metrics["recall"], 4),
                "flag_rate": round(hc_v2_master_metrics["flag_rate"], 4),
            }
        ]
    )

    top_of_funnel = pd.DataFrame(hc_v2_master_metrics["top_of_funnel"]).T.reset_index().rename(columns={"index": "bucket"})
    display(master_summary)
    display(top_of_funnel)


Loaded 26,635 rows, 63 columns
Target distribution:
TARGET
0    23747
1     2888
Default rate: 10.84%


,rows,roc_auc,avg_precision,precision,recall,flag_rate
0,26635,0.7337,0.2561,0.4114,0.0828,0.0218


,bucket,default_rate,defaults_captured,lift
0,top_50,0.4000,20.0,3.69
1,top_100,0.4400,44.0,4.06
2,top_250,0.4800,120.0,4.43
3,top_500,0.4120,206.0,3.80
4,top_1000,0.3660,366.0,3.38
5,top_2000,0.3175,635.0,2.93


## 5. Synthetic Gig-Worker Model

This section runs the synthetic gig-worker training and demo flow from `gig_credit_model.py`.


In [9]:
gig_credit_mod = load_step3_module("gig_credit_model.py", "step3_gig_credit_nb")

gig_X, gig_y = gig_credit_mod.build_training_data(n_workers=500, seed=42)
gig_metrics = gig_credit_mod.train_model(
    gig_X,
    gig_y,
    model_path=str(GIG_WORKER_MODEL),
    results_path=str(GIG_WORKER_RESULTS),
    metrics_path=str(GIG_WORKER_METRICS),
    threshold=0.5,
)

gig_summary = pd.DataFrame(
    [
        {
            "rows": gig_X.shape[0],
            "features": gig_X.shape[1],
            "cv_auc_mean": round(gig_metrics["cv_auc_mean"], 4),
            "test_roc_auc": round(gig_metrics["test_roc_auc"], 4),
            "test_avg_precision": round(gig_metrics["test_average_precision"], 4),
        }
    ]
)
gig_top_features = pd.DataFrame(
    list(gig_metrics["top_features"].items()),
    columns=["feature", "importance"],
)

display(gig_summary)
gig_top_features.head(15)


  GIG CREDIT MODEL — Building Training Data
[SYNTH] Generated 500 workers | 162,949 transactions | Default rate: 29.2%
[FEATURES] Extracted 26 features for 500 workers
[DATA] Final matrix: 500 workers x 26 features
[DATA] Default rate: 29.2% (146 positives / 500 total)

[CV] Running 5-fold stratified cross-validation...
  Fold 1: AUC = 0.7633
  Fold 2: AUC = 0.8457
  Fold 3: AUC = 0.8307
  Fold 4: AUC = 0.7953
  Fold 5: AUC = 0.8928
  OOF AUC: 0.8078 (mean fold: 0.8256 +/- 0.0441)

[TRAIN] Fitting final model on 425 rows...

[TEST] Held-out test results (75 rows):
  ROC-AUC:           0.7350
  Average Precision: 0.4955

  Model saved:   /Users/adittsingh/GigScore/STEP-3/models/gig_credit_model.joblib
  Results saved: /Users/adittsingh/GigScore/results/gig_model_validation_results.csv
  Metrics saved: /Users/adittsingh/GigScore/results/gig_model_metrics.json


,rows,features,cv_auc_mean,test_roc_auc,test_avg_precision
0,500,26,0.8256,0.735,0.4955


,feature,importance
0,avg_monthly_income,0.165906
1,total_income_6m,0.133274
2,txn_frequency_monthly,0.073636
3,monthly_surplus_ratio,0.061535
4,food_to_income_ratio,0.045420
5,income_trend,0.037962
6,rootedness_pct,0.034457
7,inflow_outflow_ratio,0.034171
8,merchant_diversity_score,0.033235
9,income_regularity_cv,0.032897


In [10]:
gig_credit_mod.run_demo(str(GIG_WORKER_MODEL))



  GIG CREDIT MODEL — Demo Scoring

  responsible_delivery.......... Score: 6.09% | Risk: VERY_LOW   | APPROVED
    Rootedness: 57.0% | FOIR: 35.6% | Income CV: 12.7% | Obligation Rate: 83%

  struggling_rider.............. Score: 36.70% | Risk: MEDIUM     | APPROVED
    Rootedness: 43.7% | FOIR: 62.7% | Income CV: 10.0% | Obligation Rate: 33%

  stable_driver................. Score: 2.09% | Risk: VERY_LOW   | APPROVED
    Rootedness: 48.1% | FOIR: 34.2% | Income CV: 14.4% | Obligation Rate: 100%

  seasonal_worker............... Score: 12.84% | Risk: VERY_LOW   | APPROVED
    Rootedness: 52.8% | FOIR: 34.3% | Income CV: 41.2% | Obligation Rate: 83%

  new_entrant................... Score: 27.44% | Risk: LOW        | APPROVED
    Rootedness: 44.5% | FOIR: 84.2% | Income CV: 19.1% | Obligation Rate: 67%

  high_earner................... Score: 2.68% | Risk: VERY_LOW   | APPROVED
    Rootedness: 68.9% | FOIR: 32.8% | Income CV: 8.9% | Obligation Rate: 100%

  chronic_defaulter...........

## 6. Single-Score Master-Dataset Model

This uses the single calibrated classifier from `gigscore_model.py`.


In [11]:
gigscore_mod = load_step3_module("gigscore_model.py", "step3_gigscore_nb")

if MASTER_DATASET_PATH is None:
    print("Skipping single-score model: master dataset not found.")
else:
    gigscore_raw = pd.read_csv(MASTER_DATASET_PATH)
    gigscore_X = gigscore_mod.build_features(gigscore_raw)
    gigscore_y = gigscore_raw["TARGET"].astype(int)

    gigscore_X_dev, gigscore_X_test, gigscore_y_dev, gigscore_y_test = train_test_split(
        gigscore_X,
        gigscore_y,
        test_size=0.15,
        random_state=gigscore_mod.SEED,
        stratify=gigscore_y,
    )
    gigscore_model, gigscore_cv, gigscore_oof = gigscore_mod.train(gigscore_X_dev, gigscore_y_dev)
    gigscore_ev = gigscore_mod.evaluate(
        gigscore_model,
        gigscore_X_test,
        gigscore_y_test,
        gigscore_cv["best_threshold"],
    )

    gigscore_summary = pd.DataFrame(
        [
            {
                "rows": len(gigscore_raw),
                "features": gigscore_X.shape[1],
                "cv_oof_auc": round(gigscore_cv["oof_auc"], 4),
                "test_auc": round(gigscore_ev["auc"], 4),
                "test_ap": round(gigscore_ev["ap"], 4),
                "threshold": round(gigscore_cv["best_threshold"], 4),
            }
        ]
    )
    gigscore_preview = pd.DataFrame(
        {
            "actual": gigscore_y_test.values,
            "default_probability": gigscore_ev["proba"],
            "predicted_label": gigscore_ev["preds"],
        },
        index=gigscore_X_test.index,
    ).head(15)

    display(gigscore_summary)
    display(gigscore_preview)



  5-fold cross-validation (22,639 rows)
    Fold 1  AUC 0.7098  AP 0.2416
    Fold 2  AUC 0.7235  AP 0.2634
    Fold 3  AUC 0.7345  AP 0.2587
    Fold 4  AUC 0.7306  AP 0.2582
    Fold 5  AUC 0.6871  AP 0.2386
    OOF    AUC 0.7094  AP 0.2346  (mean AUC 0.7171 +/- 0.0172)

  Training final model on full dev set...
  Chosen threshold: 0.5009 (P=0.213  R=0.609  F1=0.315)
  Strategy: best F1 with recall >= 50%

  Threshold sweep (calibration set):
   Thresh    Prec  Recall      F1  Flagged
     0.30  13.6%  91.8%  0.236    2,493
     0.35  14.9%  84.8%  0.253    2,100
     0.40  16.4%  79.3%  0.272    1,780
     0.45  18.3%  71.2%  0.291    1,435
     0.50  21.1%  60.9%  0.314    1,061 <--
     0.55  23.9%  50.0%  0.323      771
     0.60  27.2%  42.9%  0.333      581
     0.65  30.9%  36.1%  0.333      431
     0.70  32.8%  25.5%  0.287      287


,rows,features,cv_oof_auc,test_auc,test_ap,threshold
0,26635,28,0.7094,0.7164,0.2565,0.5009


,actual,default_probability,predicted_label
25388,0,0.492077,0
13572,0,0.418449,0
18281,0,0.328425,0
2714,0,0.182106,0
8758,0,0.377838,0
24807,0,0.269237,0
13377,0,0.446023,0
24922,0,0.768871,1
23321,0,0.399481,0
12234,0,0.284522,0


## 7. Dual-Model Pipeline

This runs the hard-reject classifier plus behavioral score regressor from `dual_model_pipeline.py`.


In [12]:
dual_model_mod = load_step3_module("dual_model_pipeline.py", "step3_dual_model_nb")

if MASTER_DATASET_PATH is None:
    print("Skipping dual-model pipeline: master dataset not found.")
else:
    dual_raw = pd.read_csv(MASTER_DATASET_PATH)
    dual_X = dual_model_mod.build_features(dual_raw)
    dual_y = dual_raw["TARGET"].astype(int)
    dual_results = dual_model_mod.run_pipeline(dual_X, dual_y)
    dual_model_mod.save_artifacts(dual_results, dual_X, dual_y)

    dual_gate_counts = dual_results["gate_df"]["decision"].value_counts().rename_axis("decision").reset_index(name="count")
    dual_summary = pd.DataFrame(
        [
            {
                "pipeline_auc": round(dual_results["pipeline_auc"], 4),
                "pipeline_ap": round(dual_results["pipeline_ap"], 4),
                "m1_test_auc": round(dual_results["m1_test_auc"], 4),
                "m2_delta_auc": round(dual_results["m2_delta_auc"], 4),
                "hard_reject_precision": round(float(dual_results["hr_precision"]), 4),
            }
        ]
    )

    display(dual_summary)
    display(dual_gate_counts)
    display(dual_results["gate_df"].head(10))



  MODEL 1: Hard-Reject Classifier
  Fold 1: AUC = 0.7095
  Fold 2: AUC = 0.7240
  Fold 3: AUC = 0.7341
  Fold 4: AUC = 0.7307
  Fold 5: AUC = 0.6867
  OOF AUC: 0.7083  (mean 0.7170 +/- 0.0173)

  Test ROC-AUC           : 0.7155
  Test Average Precision : 0.2468

  MODEL 2: Behavioral Score Regressor
  Training set: 22,639 applicants

  Behavioral delta stats (test set, n=3,996):
    MAE  : 0.1796
    RMSE : 0.3005
    Rank-order AUC (delta as risk score) : 0.7126

  Behavioral delta risk bands:
  Band           Count  Default#   Default%  Avg Delta
  RISKY             15         8     53.3%     0.453
  MODERATE         169        58     34.3%     0.635
  GOOD             667       132     19.8%     0.792
  EXCELLENT      3,145       235      7.5%     0.928

  COMBINED RISK SCORE + DUAL CONFIDENCE GATE

  HARD-REJECT GATE VALIDATION (delta < 0.75 AND flags >= 2)

  Decision distribution (n=3,996):
    HARD_REJECT :   277 (6.9%)
    SOFT_FLAG   : 1,788 (44.7%)
    PASS        : 1,931 (4

,pipeline_auc,pipeline_ap,m1_test_auc,m2_delta_auc,hard_reject_precision
0,0.6311,0.1607,0.7155,0.7126,0.296


,decision,count
0,PASS,1931
1,SOFT_FLAG,1788
2,HARD_REJECT,277


,m1_prob,m2_delta,combined_score,flag_count,delta_trigger,flag_trigger,decision,actual
0,0.472522,0.878438,0.537107,0,False,False,PASS,0
1,0.413756,0.921021,0.537192,1,False,False,PASS,0
2,0.365708,0.954920,0.653887,3,False,True,SOFT_FLAG,0
3,0.317755,0.987016,0.327287,0,False,False,PASS,0
4,0.443161,0.926142,0.484804,0,False,False,PASS,0
5,0.354344,0.951519,0.386260,0,False,False,PASS,0
6,0.456524,0.916377,0.573118,1,False,False,PASS,0
7,0.656682,0.683127,0.966496,6,True,True,HARD_REJECT,0
8,0.424483,0.925687,0.695572,3,False,True,SOFT_FLAG,0
9,0.354822,0.926581,0.402789,0,False,False,PASS,0


## 8. Behavioral Three-Zone Model

This is the broader notebook form of the existing three-zone workbench and uses `test_on_master_gig.py` directly.


In [13]:
three_zone_mod = load_step3_module("test_on_master_gig.py", "step3_three_zone_nb")

if MASTER_DATASET_PATH is None:
    print("Skipping three-zone model: master dataset not found.")
else:
    three_zone_raw = pd.read_csv(MASTER_DATASET_PATH)
    three_zone_y = three_zone_raw["TARGET"].astype(int)
    three_zone_flags = three_zone_mod.build_red_flag_frame(three_zone_raw)
    three_zone_X = three_zone_mod.build_behavioral_features(three_zone_raw, three_zone_flags)

    three_zone_feature_inventory = pd.DataFrame(
        {
            "feature": three_zone_X.columns,
            "group": [three_zone_mod.FEATURE_GROUP_LOOKUP[c] for c in three_zone_X.columns],
        }
    )
    three_zone_overview = pd.DataFrame(
        [
            {
                "rows": len(three_zone_raw),
                "behavioral_feature_count": three_zone_X.shape[1],
                "portfolio_default_rate": round(float(three_zone_y.mean()), 4),
            }
        ]
    )

    display(three_zone_overview)
    display(three_zone_feature_inventory.head(20))


,rows,behavioral_feature_count,portfolio_default_rate
0,26635,40,0.1084


,feature,group
0,ext_source_mean,Behavioral Composite
1,ext_source_min,Behavioral Composite
2,ext_source_std,Behavioral Composite
3,foir,Behavioral Composite
4,cred_savings_ratio,Behavioral Composite
5,cred_debt_income_ratio,Behavioral Composite
6,obligation_fulfillment_rate,Behavioral Composite
7,loan_default_rate,Behavioral Composite
8,income_vs_plfs_ratio,Behavioral Composite
9,income_regularity_cv,Behavioral Velocity


In [14]:
if MASTER_DATASET_PATH is None:
    print("Skipping three-zone training: master dataset not found.")
else:
    three_zone_results = three_zone_mod.train_and_evaluate(
        three_zone_X,
        three_zone_y,
        three_zone_raw,
        three_zone_flags,
        n_folds=three_zone_mod.CV_FOLDS,
    )
    three_zone_report_text = three_zone_mod.generate_report(three_zone_results)
    print(three_zone_report_text)


  Fold 1: AUC=0.7067 AP=0.2472
  Fold 2: AUC=0.7250 AP=0.2734
  Fold 3: AUC=0.7244 AP=0.2602
  Fold 4: AUC=0.7258 AP=0.2662
  Fold 5: AUC=0.6850 AP=0.2444
  GIGSCORE MASTER DATASET — BEHAVIORAL THREE-ZONE REPORT

--------------------------------------------------------------------------------
  1. FEATURE DISCIPLINE
--------------------------------------------------------------------------------
  Behavioral feature count   : 40
  Removed raw application    : loan amount, annuity amount, goods price,
                               raw credit burden ratios tied to application data
  Added rule-informed inputs : gambling severity, rent severity,
                               spend-balance severity, red-flag counts
  Feature weighting pass     : domain priors blended with dev-set signal

--------------------------------------------------------------------------------
  2. MODEL RANKING PERFORMANCE
--------------------------------------------------------------------------------
  Cross-Va

In [15]:
if MASTER_DATASET_PATH is None:
    print("Skipping three-zone diagnostics: master dataset not found.")
else:
    three_zone_diagnostic = pd.DataFrame(
        [three_zone_mod.serialize_metric(three_zone_results["diagnostic_metrics"])]
    )
    three_zone_summary = pd.DataFrame(three_zone_results["zone_metrics"]["zones"]).T
    three_zone_top_features = pd.DataFrame(three_zone_results["top_features"]).head(15)
    three_zone_agent4_payload = three_zone_mod.build_agent4_payload(
        three_zone_results["raw_test"],
        three_zone_results["zone_frame"],
        three_zone_results["portfolio_safe_rate"],
    )

    display(three_zone_diagnostic)
    display(three_zone_summary)
    display(three_zone_top_features)
    three_zone_agent4_payload["applicants"][:3]


,threshold,precision,recall,f1,accuracy,specificity,flag_rate,tn,fp,fn,tp
0,0.28,0.22551,0.510393,0.31281,0.757007,0.786977,0.245245,2804,759,212,221


,count,share,default_rate,avg_model_confidence,avg_combined_risk_score,avg_gigscore,avg_behavioral_delta,avg_score_adjustment,avg_confirmed_red_flags
HARD_REJECT,11.0,0.002753,0.545455,0.559950,0.910476,8.952363,-10.000000,-10.000000,2.000000
CAUTION,1360.0,0.340340,0.199265,0.341766,0.741771,25.822915,-5.607141,-5.622794,1.630882
CLEAR,2625.0,0.656907,0.059429,0.165375,0.546246,45.375431,4.298878,4.306286,1.574857


,name,group,raw_importance,feature_weight,weighted_importance,signal_auc
0,ext_source_mean,Behavioral Composite,0.365705,2.278125,0.833122,0.711007
1,ext_source_min,Behavioral Composite,0.057117,1.966738,0.112334,0.685528
2,employment_age_ratio,Behavioral Stability,0.046383,0.911239,0.042266,0.561689
3,ext_score_x_stability,Derived Interactions,0.046178,0.911763,0.042103,0.639587
4,years_employed,Behavioral Stability,0.037467,0.962278,0.036053,0.574431
5,age,Behavioral Stability,0.039495,0.845677,0.033400,0.555878
6,ext_source_std,Behavioral Composite,0.038459,0.858595,0.033021,0.536013
7,income_vs_plfs_ratio,Behavioral Composite,0.033903,0.767184,0.026010,0.505512
8,obligation_fulfillment_rate,Behavioral Composite,0.031876,0.647199,0.020630,0.504223
9,ext_score_x_foir,Derived Interactions,0.031101,0.646264,0.020100,0.570341


## 9. Charts and CLI Fallbacks

`generate_report_charts.py` expects `results/master_dataset_predictions.csv`. If that file already exists, the next cell will regenerate the chart pack. The final cell lists the exact CLI reruns for every Step 3 script.


In [16]:
charts_mod = load_step3_module("generate_report_charts.py", "step3_charts_nb")

if MASTER_PREDICTIONS.exists():
    charts_mod.main()
    print(f"Charts regenerated in {CHARTS_DIR}")
else:
    print(f"Skipping chart generation: missing {MASTER_PREDICTIONS}")


  GENERATING REPORT CHARTS

  Loaded 22,182 predictions

  [1] ROC Curve
  [2] Precision-Recall Curve
  [3] Confusion Matrix
  [4] Risk Band Distribution
  [5] Score Distribution
  [6] Cumulative Gains & Lift Chart
  [7] Top-of-Funnel Chart
  [8] Threshold Sensitivity
  [9] Train vs Test Comparison
  [10] Feature Distributions

  All 10 charts saved to /Users/adittsingh/GigScore/results/charts/
Charts regenerated in /Users/adittsingh/GigScore/results/charts


In [17]:
cli_recipes = {
    "legacy_train": f"{sys.executable} STEP-3/train.py",
    "legacy_predict_demo": f"{sys.executable} STEP-3/predict.py --demo --user-id USR0005",
    "home_credit_baseline": f"{sys.executable} STEP-3/home_credit_task3.py",
    "home_credit_improved": f"{sys.executable} STEP-3/home_credit_improved.py",
    "home_credit_master_test": f"{sys.executable} STEP-3/test_on_master.py",
    "gig_worker_model": f"{sys.executable} STEP-3/gig_credit_model.py --mode both",
    "single_score_master": f"{sys.executable} STEP-3/gigscore_model.py",
    "dual_model_pipeline": f"{sys.executable} STEP-3/dual_model_pipeline.py",
    "behavioral_three_zone": f"{sys.executable} STEP-3/test_on_master_gig.py",
    "report_charts": f"{sys.executable} STEP-3/generate_report_charts.py",
}

pd.DataFrame(
    [{"workflow": key, "command": value} for key, value in cli_recipes.items()]
)


,workflow,command
0,legacy_train,/opt/anaconda3/bin/python STEP-3/train.py
1,legacy_predict_demo,/opt/anaconda3/bin/python STEP-3/predict.py --...
2,home_credit_baseline,/opt/anaconda3/bin/python STEP-3/home_credit_t...
3,home_credit_improved,/opt/anaconda3/bin/python STEP-3/home_credit_i...
4,home_credit_master_test,/opt/anaconda3/bin/python STEP-3/test_on_maste...
5,gig_worker_model,/opt/anaconda3/bin/python STEP-3/gig_credit_mo...
6,single_score_master,/opt/anaconda3/bin/python STEP-3/gigscore_mode...
7,dual_model_pipeline,/opt/anaconda3/bin/python STEP-3/dual_model_pi...
8,behavioral_three_zone,/opt/anaconda3/bin/python STEP-3/test_on_maste...
9,report_charts,/opt/anaconda3/bin/python STEP-3/generate_repo...
